In [ ]:
# Cell 1: Install arc-agi from competition wheels
import subprocess, sys
wheels = '/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels'
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    '--no-index', f'--find-links={wheels}',
    'arc-agi', 'python-dotenv'
])
print('[Cell1] arc-agi installed')


In [ ]:
# Cell 2: DenseExplorer BFS + universal adapter + diagnostic
import os, hashlib, collections, random, json, time, gc, traceback, inspect
import numpy as np
import pandas as pd
from arcengine import GameAction

# ── Universal adapter (works with any arc-agi version) ──
def _reset(env):
    r = env.reset()
    if isinstance(r, tuple):
        return r[0]
    return r

def _step(env, action, x=None, y=None):
    if hasattr(action, 'is_complex') and action.is_complex():
        if x is None or y is None:
            return None
        try:
            a = GameAction(int(action))
            a.set_data({'x': int(x), 'y': int(y)})
            return env.step(a)
        except Exception:
            return None
    try:
        return env.step(action)
    except Exception:
        return None

def _grid(frame):
    fr = getattr(frame, 'frame', None)
    if fr is not None and len(fr) > 0:
        return np.asarray(fr[0], dtype=np.int32)
    return np.zeros((64, 64), dtype=np.int32)

def _hash_grid(grid):
    import hashlib as _hl
    return int(_hl.md5(np.asarray(grid, dtype=np.int32).tobytes()).hexdigest()[:16], 16)

def _is_win(frame):
    s = getattr(frame, 'state', None)
    return s is not None and 'WIN' in str(s)

def _get_action_list(env):
    for attr in ('actions', 'action_space'):
        try:
            acts = list(getattr(env, attr))
            if acts and len(acts) > 0 and hasattr(acts[0], 'action_type'):
                return acts
        except Exception:
            continue
    return []

_ALL_POS = [(x, y) for y in range(0, 64, 2) for x in range(0, 64, 2)]

# ── DenseExplorer (direct env loop, 3 phases) ──
class DenseExplorer:
    def __init__(self, env, actions):
        self._env = env
        self._acts = actions
        self._cidx = next((i for i, a in enumerate(actions) if getattr(a, 'is_complex', lambda: False)() or hasattr(a, 'is_complex') and a.is_complex()), None)
        self._sidx = [i for i, a in enumerate(actions) if not (getattr(a, 'is_complex', lambda: False)() or hasattr(a, 'is_complex') and a.is_complex())]
        self._budget = 0
        self._steps = 0
        self.solution = None
        self.live = []
    def _stp(self, ai, cx=0, cy=0):
        self._steps += 1
        return _step(self._env, self._acts[ai], cx, cy)
    def _ph1(self, budget):
        for ai in self._sidx:
            if self._steps >= budget:
                return None
            _reset(self._env)
            for k in range(200):
                f = self._stp(ai)
                if f is None: break
                if _is_win(f): return [ai] * (k + 1)
                if self._steps >= budget: break
        return None
    def _ph2(self, maxp=1024):
        live = []
        _reset(self._env)
        bf = self._stp(self._sidx[0]) if self._sidx else (self._stp(self._cidx, 0, 0) if self._cidx is not None else None)
        bh = _hash_grid(_grid(bf)) if bf else 0
        for pi, (px, py) in enumerate(_ALL_POS):
            if pi >= maxp or self._steps >= self._budget: break
            _reset(self._env)
            if self._cidx is None: continue
            f = self._stp(self._cidx, px, py)
            if f is None: continue
            if _is_win(f):
                self.solution = [(self._cidx, px, py)]
                return 'WIN', [(px, py, 0)]
            h = _hash_grid(_grid(f))
            if h != bh:
                live.append((px, py, h))
        return ('LIVE' if live else 'NONE', live)
    def _ph3(self, live, maxs):
        if not live: return False
        seen = set()
        frontier = [(h, [(self._cidx, px, py)], [(px, py)]) for px, py, h in live if h not in seen and (seen.add(h) or True)]
        explored = set(seen)
        while frontier and self._steps < maxs:
            frontier.sort(key=lambda n: len(n[1]))
            ch, prefix, xyl = frontier.pop(0)
            _reset(self._env)
            ok = all(self._stp(ai, cx, cy) is not None for ai, cx, cy in prefix)
            if not ok: continue
            for ai in self._sidx:
                if self._steps >= maxs: return self.solution is not None
                f = self._stp(ai)
                if f is None: continue
                if _is_win(f):
                    self.solution = [a for a, _, _ in prefix] + [ai]
                    return True
                nh = _hash_grid(_grid(f))
                if nh not in explored:
                    explored.add(nh)
                    frontier.append((nh, list(prefix) + [(ai, 32, 32)], xyl))
            for lcx, lcy in set((x, y) for x, y, _ in live):
                if self._cidx is None or self._steps >= maxs: break
                f = self._stp(self._cidx, lcx, lcy)
                if f is None: continue
                if _is_win(f):
                    self.solution = [a for a, _, _ in prefix] + [(self._cidx, lcx, lcy)]
                    return True
                nh = _hash_grid(_grid(f))
                if nh not in explored:
                    explored.add(nh)
                    frontier.append((nh, list(prefix) + [(self._cidx, lcx, lcy)], [(lcx, lcy)]))
        return self.solution is not None
    def explore(self, maxs=2000):
        self._budget = maxs
        self._steps = 0
        self.solution = None
        self.live = []
        if self._sidx:
            s = self._ph1(maxs)
            if s: self.solution = s; return True
        if self._cidx is not None and self._steps < self._budget:
            r, d = self._ph2(1024)
            if r == 'WIN': return True
            if r == 'LIVE': self.live = d
        if self.live and self._steps < self._budget:
            self._ph3(self.live, maxs // 2)
        return self.solution is not None

# ── STEP 1: API Diagnostic (runs in Phase A AND Phase B) ──
print('=== API DIAGNOSTIC ===')
try:
    from arc_agi import Arcade
    print(f'Arcade import: OK | type={type(Arcade)}')
    arcade = Arcade()
    print(f'Arcade(): OK | dir={[x for x in dir(arcade) if not x.startswith("_")][:20]}')
    # Try to get environments
    for attr in ['get_environments', 'available_environments', 'environments', 'list_games']:
        if hasattr(arcade, attr) or hasattr(Arcade, attr):
            src = getattr(arcade, attr, None) or getattr(Arcade, attr, None)
            print(f'  {attr}: {type(src)} callable={callable(src)}')
            if callable(src):
                try: games = src() if hasattr(arcade, attr) else (src() if callable(src) else src)
                except: print(f'    call FAILED: {traceback.format_exc()[:200]}')
    # Try creating an env if any games available
    try:
        games = arcade.get_environments() if hasattr(arcade, 'get_environments') else getattr(Arcade, 'available_environments', [])
        if not games:
            # Try directly
            for gattr in ['game_ids', 'game_id_list']:
                if hasattr(arcade, gattr):
                    games = getattr(arcade, gattr)
                    break
        if games:
            print(f'Games: {len(games)} first={games[0]}')
    except Exception as e:
        print(f'Games list failed: {e}')
    # Try inspect env
    try:
        gid = str(games[0]) if hasattr(games[0], 'game_id') else str(games[0])
        env = arcade.make(gid) if hasattr(arcade, 'make') else Arcade.make(gid)
        print(f'make_env: OK | env type={type(env).__name__}')
        fr = _reset(env)
        print(f'reset: OK | type={type(fr).__name__} | has frame={hasattr(fr, "frame")} | frame[0] shape={np.array(fr.frame[0]).shape if hasattr(fr, "frame") and fr.frame else "N/A"}')
        acts = getattr(fr, 'available_actions', [])
        print(f'available_actions: {len(acts)} | first type={type(acts[0]).__name__ if acts else "NONE"}')
        if acts:
            a0 = acts[0]
            print(f'  action attrs: {[x for x in dir(a0) if not x.startswith("_")][:10]}')
            print(f'  action_type: {getattr(a0, "action_type", "N/A")}')
            print(f'  is_complex: {hasattr(a0, "is_complex")}')
            if hasattr(a0, 'is_complex'):
                print(f'  is_complex(): {a0.is_complex() if callable(a0.is_complex) else a0.is_complex}')
                print(f'  set_data: {hasattr(a0, "set_data")}')
        env.close()
    except Exception as e:
        print(f'env test FAILED: {e}')
        traceback.print_exc()
except Exception as e:
    print(f'Arcade import FAILED: {e}')

# ── STEP 2: Framework fallback agent (written to /tmp/my_agent.py) ──


In [ ]:
# Cell 3: Phase B — Competition Rerun (DenseExplorer direct)
import os, subprocess, shutil, sys, time, json as _json

if os.environ.get('KAGGLE_IS_COMPETITION_RERUN'):
    print('[Cell3] Phase B: competition rerun detected')

    # Wait for gateway
    subprocess.run(['curl', '--fail', '--retry', '30', '--retry-delay', '1', '--max-time', '5', 'http://gateway:8001/api/games'])

    results = {}
    try:
        from arc_agi import Arcade
        arc = Arcade()
        # Find games
        games = []
        if hasattr(arc, 'get_environments'):
            games = arc.get_environments()
        elif hasattr(Arcade, 'available_environments'):
            games = Arcade.available_environments

        if not games:
            print('[Cell3] No games found via Arcade API')
            # Manual scan
            import glob as _gl
            for p in _gl.glob('/kaggle/input/**/*.json', recursive=True):
                try:
                    with open(p) as f: md = _json.load(f)
                    gid = md.get('game_id', md.get('id', ''))
                    if gid and len(str(gid)) > 5:
                        games.append(str(gid))
                except: pass
            games = list(set(games))

        print(f'[Cell3] {len(games)} games found')
        results = {}

        for g in games:
            gid = g.game_id if hasattr(g, 'game_id') else str(g)
            print(f'  [{gid}] start...', flush=True)
            t0 = time.time()

            try:
                env = arc.make(gid) if hasattr(arc, 'make') else Arcade.make(gid)
                acts = _get_action_list(env)
                if not acts:
                    print(f'  [{gid}] no actions')
                    results[gid] = 0.0
                    env.close()
                    continue

                dex = DenseExplorer(env, acts)
                found = dex.explore(maxs=5000)

                if found and dex.solution:
                    _reset(env)
                    score = 0.0
                    for item in dex.solution:
                        if isinstance(item, (list, tuple)) and len(item) >= 3:
                            ai, cx, cy = int(item[0]), int(item[1]), int(item[2])
                        else:
                            ai, cx, cy = int(item), 32, 32
                        if ai >= len(acts): break
                        f = _step(env, acts[ai], cx, cy)
                        if f is None: break
                        score += float(getattr(f, 'reward', getattr(f, 'score', 0.0)))
                        if getattr(f, 'done', False): break
                    results[gid] = score
                    print(f'  [{gid}] WIN sc={score:.4f} t={time.time()-t0:.1f}s', flush=True)
                else:
                    results[gid] = 0.0
                    print(f'  [{gid}] no solution t={time.time()-t0:.1f}s', flush=True)

                env.close()
            except Exception as e:
                print(f'  [{gid}] ERROR: {e}', flush=True)
                results[gid] = 0.0

        # Write submission
        solved = sum(1 for v in results.values() if v > 0.0)
        mean = sum(results.values()) / max(1, len(results))
        print(f'[Cell3] RESULT: {solved}/{len(results)} solved mean={mean:.4f}', flush=True)

        rows = [{'row_id': f'{k}_0', 'game_id': str(k), 'end_of_game': True, 'score': v} for k, v in results.items()]
        pd.DataFrame(rows).to_parquet('/kaggle/working/submission.parquet', index=False)
        print(f'[Cell3] submission.parquet written ({len(rows)} games)', flush=True)

    except Exception as e:
        print(f'[Cell3] FATAL: {e}', flush=True)
        traceback.print_exc()
        # Emergency fallback: write empty submission
        pd.DataFrame([{'row_id': '1_0', 'game_id': '1', 'end_of_game': True, 'score': 0.0}]).to_parquet('/kaggle/working/submission.parquet', index=False)
        print('[Cell3] emergency submission.parquet written', flush=True)
else:
    print('[Cell3] Phase A: skipping (no competition rerun)')


In [ ]:
# Cell 4: Phase A — Write dummy submission.parquet
import os
import pandas as pd

if not os.environ.get('KAGGLE_IS_COMPETITION_RERUN'):
    print('[Cell4] Phase A: writing dummy submission.parquet')
    pd.DataFrame(
        data=[['1_0', '1', True, 1]],
        columns=['row_id', 'game_id', 'end_of_game', 'score']
    ).to_parquet('/kaggle/working/submission.parquet', index=False)
    print('[Cell4] submission.parquet written — click Submit to Competition on Kaggle')
else:
    print('[Cell4] Phase B: submission.parquet handled by Cell 3')
